# RAG Demo: Filings Question Answering

This notebook demonstrates a read-only Retrieval-Augmented Generation (RAG) pipeline for investment research QA using precomputed filings artifacts. It inspects existing data products, performs deterministic retrieval, and synthesizes a cited answer without recomputing any embeddings or indexes.

## Environment setup

In [ ]:
# --- Notebook environment setup ---
# 1) Enable imports from local src/
# 2) Normalize working directory so relative paths work consistently

import sys
import os
from pathlib import Path

# Resolve project root (repo root)
PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

# Add src/ to Python path
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Normalize working directory (critical for relative paths in library code)
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Src path added:", SRC_PATH)
print("Working directory:", os.getcwd())

In [ ]:
# --- Runtime secrets (notebook-safe) ---
# Required for retrieval / answer synthesis

import os
from dotenv import load_dotenv

env_path = PROJECT_ROOT / ".env"

if env_path.exists():
    load_dotenv(env_path)
    print("Loaded environment variables from .env")
else:
    print("No .env file found; relying on existing environment")

# Sanity check (does NOT print the key)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set for notebook runtime"

## Imports and Artifact Paths

In [ ]:
import json
import pandas as pd
import pyarrow.parquet as pq

# Reuse PROJECT_ROOT from environment setup cell (DO NOT redefine)
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "filings" / "NU" / "Q2"

sections_path   = DATA_DIR / "sections.parquet"
chunks_path     = DATA_DIR / "chunks.parquet"
embeddings_path = DATA_DIR / "embeddings.parquet"
manifest_path   = DATA_DIR / "embeddings_manifest.json"

## Artifact Sanity Check 

In [ ]:
# PROJECT_ROOT is already defined and cwd is already set correctly
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "filings" / "NU" / "Q2"

sections_path   = DATA_DIR / "sections.parquet"
chunks_path     = DATA_DIR / "chunks.parquet"
embeddings_path = DATA_DIR / "embeddings.parquet"
manifest_path   = DATA_DIR / "embeddings_manifest.json"

if not sections_path.exists():
    print("Note: sections.parquet not found (optional for this demo).")

assert chunks_path.exists(), f"Missing: {chunks_path}"
assert embeddings_path.exists(), f"Missing: {embeddings_path}"
assert manifest_path.exists(), f"Missing: {manifest_path}"

with open(manifest_path, "r", encoding="utf-8") as handle:
    manifest = json.load(handle)

print(json.dumps(manifest, indent=2, sort_keys=True))

## Inspect Chunking

We load the chunked sections dataset and summarize chunk counts. The example chunk text is truncated for display only.

In [ ]:
chunks_table = pq.read_table(chunks_path)
chunks_df = chunks_table.to_pandas()

total_chunks = len(chunks_df)
doc_type_counts = chunks_df.groupby("doc_type").size().reset_index(name="count")

print(f"Total chunks: {total_chunks}")
display(doc_type_counts)

example_text = chunks_df.iloc[0]["text"] if total_chunks else ""
print(example_text[:500] + ("..." if len(example_text) > 500 else ""))

## Inspect Embeddings

We verify that embeddings align with chunks and check embedding dimensionality.

In [ ]:
embeddings_table = pq.read_table(embeddings_path)
embeddings_df = embeddings_table.to_pandas()

print(f"Embeddings: {len(embeddings_df)}")
print(f"Chunks: {len(chunks_df)}")
assert len(embeddings_df) == len(chunks_df), "Embeddings count does not match chunks."

embedding_dim = embeddings_df.iloc[0]["embedding_dim"] if len(embeddings_df) else None
sample_vector = embeddings_df.iloc[0]["embedding"] if len(embeddings_df) else []
print(f"Embedding dim: {embedding_dim}")
print(f"Sample vector length: {len(sample_vector)}")
assert embedding_dim == len(sample_vector), "Embedding dimension mismatch in sample."

## Retrieval Demo

We run deterministic retrieval over the existing embeddings. Results are displayed with metadata for traceability.

In [ ]:
from genai_filings.retrieval import retrieve

query = "How is net interest margin evolving?"
results = retrieve(query=query, issuer="NU", period="Q2", k=5)

results_df = pd.DataFrame(results)[
    ["rank", "score", "doc_type", "source_file", "chunk_id", "text_preview"]
]
display(results_df)

## Answer Synthesis Demo

We synthesize a cited answer using only the retrieved chunks, preserving citations and provenance.

In [ ]:
from genai_filings.answering import synthesize_answer
from IPython.display import Markdown, display

answer = synthesize_answer(
    query=query,
    issuer="NU",
    period="Q2",
    k=5,
    model="gpt-4.1",
    temperature=0.0,
    max_tokens=500,
)

display(Markdown(answer["answer_markdown"]))

## Traceability

- Each answer is grounded in retrieved chunks that include issuer, period, document type, source file, and chunk ID.
- Hallucination is structurally prevented by restricting the model to the provided excerpts and requiring explicit citation by source number.
- Citations map directly to chunk identifiers, allowing audit back to the original filings PDFs.

## Conclusion

This capsule demonstrates a deterministic, read-only RAG workflow for filings-based QA: inspection, retrieval, and cited synthesis. It intentionally excludes agents, memory, evaluation harnesses, or any recomputation of embeddings or indexes.